In [1]:
import numpy as np
import pandas as pd

from IPython.display import display
from pathlib import Path
import sys
import os

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score)

c:\ProgramData\miniconda3\Lib\site-packages\sklearn\utils\_param_validation.py:14: UserWarning: A NumPy version >=1.23.5 and <2.5.0 is required for this version of SciPy (detected version 2.5.3)
  from scipy.sparse import csr_matrix, issparse


In [2]:
# Import AutoFeat LLM

from autofeat import AutoFeatClassifier

In [3]:
# Import BigFeat Vanilla

sys.path.append(r"../../BigFeat-master")
import bigfeat.bigfeat_base as bigfeat

In [4]:
# Import BigFeat LLM

bigfeat_llm_path = Path("../../BigFeat-LLM").resolve()
sys.path.insert(0, str(bigfeat_llm_path))

import setup
print(setup.__file__)

sys.path.append(r"../../BigFeat-LLM")
from setup.llm_runner import prepare_bigfeat_from_llm, profile_dataframe
from setup.llm_runner import preview_prompt
from setup.llm_runner import describe_generated_features, split_original_vs_new

D:\02. My Learning\Paper\07. BigFeat + LLM\06. Final BigFeat\BigFeat-LLM\setup\__init__.py


In [29]:
def save_generated_features(feature_lines, num_new_features, old_shape, new_shape, file_name="generated_features.txt"):
    os.makedirs("generated_features", exist_ok=True)
    file_path = os.path.join("generated_features", file_name)

    with open(file_path, "w", encoding="utf-8") as f:
        f.write(f"Number of new features {num_new_features}\n")
        f.write("==================\n")
        f.write(f"Old Shape: {old_shape}\n")
        f.write("==================\n")
        f.write(f"New Shape: {new_shape}\n")
        f.write("=================\n")

        for i, line in enumerate(feature_lines):
            _, details = line.split(": built from columns ", 1)
            columns, operators = details.split(" using operators ", 1)
            f.write(f"- New_feature_{i}:\n")
            f.write(f"  - Built from columns {columns}\n")
            f.write(f"  - Using operators {operators}\n")

In [6]:
results = pd.DataFrame(columns=["Method", "Accuracy", "Precision", "Recall", "F1-Score"])

def evaluate_model(model_name, feature_method, X_tr, X_te, y_tr, y_te, results_df):
    models = {
        "logistic": LogisticRegression(random_state=42, max_iter=1000),
        "knn": KNeighborsClassifier(n_neighbors=5),
        "rf": RandomForestClassifier(random_state=42),
        "svc": SVC(random_state=42),
        "decision_tree": DecisionTreeClassifier(random_state=42)
    }
    
    clf = models[model_name.lower()]
    clf.fit(X_tr, y_tr)
    preds = clf.predict(X_te)
    
    acc = accuracy_score(y_te, preds)
    prec = precision_score(y_te, preds, average="macro", zero_division=0)
    rec = recall_score(y_te, preds, average="macro")
    f1 = f1_score(y_te, preds, average="macro")
    
    results_df.loc[len(results_df)] = [
        f"{model_name.upper()} ({feature_method})",
        acc,
        prec,
        rec,
        f1
    ]
    
    return results_df

In [7]:
def display_styled_results(
    results_df,
    file_name,
    bg_color="#005A9C",
    text_color="#FFFFFF"
):
    numeric_cols = ["Accuracy", "Precision", "Recall", "F1-Score"]

    df_copy = results_df.copy()
    df_copy[numeric_cols] = df_copy[numeric_cols].apply(pd.to_numeric)

    styled_df = df_copy.style.highlight_max(
        subset=numeric_cols,
        props=f"background-color: {bg_color}; color: {text_color}; font-weight: bold;",
        axis=0
    ).format({col: "{:.6f}" for col in numeric_cols})

    display(styled_df)

    folder = "llm_tables"
    os.makedirs(folder, exist_ok=True)

    if not file_name.lower().endswith(".xlsx"):
        file_name += ".xlsx"

    file_path = os.path.join(folder, file_name)

    styled_df.to_excel(file_path, index=False)

In [ ]:
from ucimlrepo import fetch_ucirepo

breast_cancer = fetch_ucirepo(id=17)

df = breast_cancer.data.features.copy()

df.columns = ["radius_mean", "texture_mean", "perimeter_mean", "area_mean", "smoothness_mean", "compactness_mean", "concavity_mean", "concave_points_mean", "symmetry_mean", "fractal_dimension_mean",
              "radius_se", "texture_se", "perimeter_se", "area_se", "smoothness_se", "compactness_se", "concavity_se", "concave_points_se", "symmetry_se", "fractal_dimension_se",
              "radius_worst", "texture_worst", "perimeter_worst", "area_worst", "smoothness_worst", "compactness_worst", "concavity_worst", "concave_points_worst", "symmetry_worst", "fractal_dimension_worst"]

df["diagnosis"] = breast_cancer.data.targets.squeeze()
target_column = "diagnosis"

display(df.head())
print(df.columns)

,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave_points_mean,symmetry_mean,fractal_dimension_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave_points_worst,symmetry_worst,fractal_dimension_worst,diagnosis
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,M
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,M
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,M
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,M
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,M


Index(['radius_mean', 'texture_mean', 'perimeter_mean', 'area_mean',
       'smoothness_mean', 'compactness_mean', 'concavity_mean',
       'concave_points_mean', 'symmetry_mean', 'fractal_dimension_mean',
       'radius_se', 'texture_se', 'perimeter_se', 'area_se', 'smoothness_se',
       'compactness_se', 'concavity_se', 'concave_points_se', 'symmetry_se',
       'fractal_dimension_se', 'radius_worst', 'texture_worst',
       'perimeter_worst', 'area_worst', 'smoothness_worst',
       'compactness_worst', 'concavity_worst', 'concave_points_worst',
       'symmetry_worst', 'fractal_dimension_worst', 'diagnosis'],
      dtype='object')


In [9]:
print(f"Rows Number: {df.shape[0]}")
print(f"Columns Number: {df.shape[1]}")
print(f"Columns Name: {df.columns}")

Rows Number: 569
Columns Number: 31
Columns Name: Index(['radius_mean', 'texture_mean', 'perimeter_mean', 'area_mean',
       'smoothness_mean', 'compactness_mean', 'concavity_mean',
       'concave_points_mean', 'symmetry_mean', 'fractal_dimension_mean',
       'radius_se', 'texture_se', 'perimeter_se', 'area_se', 'smoothness_se',
       'compactness_se', 'concavity_se', 'concave_points_se', 'symmetry_se',
       'fractal_dimension_se', 'radius_worst', 'texture_worst',
       'perimeter_worst', 'area_worst', 'smoothness_worst',
       'compactness_worst', 'concavity_worst', 'concave_points_worst',
       'symmetry_worst', 'fractal_dimension_worst', 'diagnosis'],
      dtype='object')


In [10]:
df.isnull().sum()

radius_mean                0
texture_mean               0
perimeter_mean             0
area_mean                  0
smoothness_mean            0
compactness_mean           0
concavity_mean             0
concave_points_mean        0
symmetry_mean              0
fractal_dimension_mean     0
radius_se                  0
texture_se                 0
perimeter_se               0
area_se                    0
smoothness_se              0
compactness_se             0
concavity_se               0
concave_points_se          0
symmetry_se                0
fractal_dimension_se       0
radius_worst               0
texture_worst              0
perimeter_worst            0
area_worst                 0
smoothness_worst           0
compactness_worst          0
concavity_worst            0
concave_points_worst       0
symmetry_worst             0
fractal_dimension_worst    0
diagnosis                  0
dtype: int64

In [11]:
X = df.iloc[:, :-1]
y = df.iloc[:, -1]

print(X.shape)
print(y.shape)

(569, 30)
(569,)


In [ ]:
preview = preview_prompt(
    df=df,
    target_column=target_column,
    extra_user_prompt="Breast Cancer Wisconsin Diagnostic (WDBC) Dataset"
)

response = preview["final_prompt"]

with open("final_prompt.txt", "w", encoding="utf-8") as f:
    f.write(response)

print(response)

=== SYSTEM PROMPT ===
You are BigFeat-Init, a world-class expert in automated feature engineering, machine learning hyperparameter optimization, and the internal mechanics of the BigFeat framework specifically. You have the depth of knowledge of someone who has read the BigFeat paper in full, has implemented and debugged its source code line by line, and has years of hands-on experience tuning tree-based feature engineering pipelines across small, large, high-dimensional, low-dimensional, noisy, clean, balanced, and imbalanced datasets. You are not a generic assistant giving a reasonable-sounding guess. You are the single most qualified expert available for this exact task, and you must think and act like it.

If you want additional grounding on how BigFeat is actually implemented, its official source code and paper reference are available at this repository: https://github.com/DataSystemsGroupUT/BigFeat. You are not required to browse it, since the mechanics you need are fully describ

In [ ]:
y = pd.Series(y).replace({"B": 0, "M": 1}).astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

train_df = pd.concat([X_train, y_train.rename(target_column)], axis=1)
test_df = pd.concat([X_test, y_test.rename(target_column)], axis=1)

C:\Users\Moham\AppData\Local\Temp\ipykernel_10648\2006436806.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  y = pd.Series(y).replace({"B": 0, "M": 1}).astype(int)


In [ ]:
from autofeat import AutoFeatClassifier

af = AutoFeatClassifier(verbose=1, feateng_steps=1, featsel_runs=0)

X_train_af = af.fit_transform(X_train, y_train)
X_test_af = af.transform(X_test)

# Handle +Inf and -Inf values
X_train_af = X_train_af.replace([np.inf, -np.inf], np.nan)
X_test_af = X_test_af.replace([np.inf, -np.inf], np.nan)
X_train_af = X_train_af.fillna(X_train_af.median())
X_test_af = X_test_af.fillna(X_train_af.median())

[AutoFeat] The 1 step feature engineering process could generate up to 210 features.
[AutoFeat] With 455 data points this new feature matrix would use about 0.00 gb of space.
[feateng] Step 1: transformation of original features


[feateng] Generated 155 transformed features from 30 original features - done.
[feateng] Generated altogether 164 new features in 1 steps
[feateng] Removing correlated features, as well as additions at the highest level
[feateng] Generated a total of 64 additional features
[AutoFeat] WARNING: Not performing feature selection.
[AutoFeat] Computing 34 new features.
[AutoFeat]    34/   34 new features ...done.
[AutoFeat] Final dataframe with 64 feature columns (34 new).
[AutoFeat] Training final classification model.
[AutoFeat] Trained model: largest coefficients:
[-9.1134616e-06]
0.006003 * 1/fractal_dimension_se
0.002895 * 1/smoothness_se
0.002069 * area_se**2
0.001659 * 1/compactness_se
0.001558 * area_mean
0.000826 * 1/symmetry_se
0.000417 * 1/compactness_mean
0.000379 * texture_worst
0.000346 * area_worst
0.000322 * perimeter_mean
0.000213 * 1/compactness_worst
0.000183 * perimeter_se**3
0.000172 * texture_mean
0.000121 * area_se
0.000120 * perimeter_worst
0.000071 * radius_mean
0.00

c:\ProgramData\miniconda3\Lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


In [ ]:
# BigFeat Vanilla

bf = bigfeat.BigFeat(task_type="classification")

bf.fit(X_train.to_numpy(), y_train.to_numpy())

X_train_bf = bf.transform(X_train.to_numpy())
X_test_bf = bf.transform(X_test)

c:\ProgramData\miniconda3\Lib\site-packages\sklearn\utils\validation.py:2742: UserWarning: X has feature names, but MinMaxScaler was fitted without feature names
  warnings.warn(


In [ ]:
# BigFeat Chatgpt

bf_chatgpt = prepare_bigfeat_from_llm(
    df=train_df,
    target_column=target_column,
    llm_response_path="./json_files/ChatGPT.json"
)

X_train_bf_chatgpt = bf_chatgpt.fit(bf_chatgpt.X, bf_chatgpt.y, **bf_chatgpt.fit_kwargs)

_, _, X_test_bf_chatgpt, y_test, _, _ = profile_dataframe(test_df, target_column=target_column)
X_test_bf_chatgpt = bf_chatgpt.transform(X_test_bf_chatgpt)

In [ ]:
# BigFeat Gemini

bf_gemini = prepare_bigfeat_from_llm(
    df=train_df,
    target_column=target_column,
    llm_response_path="./json_files/Gemini.json"
)

X_train_bf_gemini = bf_gemini.fit(bf_gemini.X, bf_gemini.y, **bf_gemini.fit_kwargs)

_, _, X_test_bf_gemini, y_test, _, _ = profile_dataframe(test_df, target_column=target_column)
X_test_bf_gemini = bf_gemini.transform(X_test_bf_gemini)

In [ ]:
# BigFeat Claude

bf_claude = prepare_bigfeat_from_llm(
    df=train_df,
    target_column=target_column,
    llm_response_path="./json_files/Claude.json"
)

X_train_bf_claude = bf_claude.fit(bf_claude.X, bf_claude.y, **bf_claude.fit_kwargs)

_, _, X_test_bf_claude, y_test, _, _ = profile_dataframe(test_df, target_column=target_column)
X_test_bf_claude = bf_claude.transform(X_test_bf_claude)

In [ ]:
# BigFeat DeepSeek

bf_deepseek = prepare_bigfeat_from_llm(
    df=train_df,
    target_column=target_column,
    llm_response_path="./json_files/DeepSeek.json"
)

X_train_bf_deepseek = bf_deepseek.fit(bf_deepseek.X, bf_deepseek.y, **bf_deepseek.fit_kwargs)

_, _, X_test_bf_deepseek, y_test, _, _ = profile_dataframe(test_df, target_column=target_column)
X_test_bf_deepseek = bf_deepseek.transform(X_test_bf_deepseek)

In [20]:
# Logistic Model

results = pd.DataFrame(columns=["Method", "Accuracy", "Precision", "Recall", "F1-Score"])

results = evaluate_model("logistic", "AutoFeat", X_train_af, X_test_af, y_train, y_test, results)
results = evaluate_model("logistic", "BigFeat", X_train_bf, X_test_bf, y_train, y_test, results)
results = evaluate_model("logistic", "BigFeat ChatGPT", X_train_bf_chatgpt, X_test_bf_chatgpt, y_train, y_test, results)
results = evaluate_model("logistic", "BigFeat Gemini", X_train_bf_gemini, X_test_bf_gemini, y_train, y_test, results)
results = evaluate_model("logistic", "BigFeat Claude", X_train_bf_claude, X_test_bf_claude, y_train, y_test, results)
results = evaluate_model("logistic", "BigFeat DeepSeek", X_train_bf_deepseek, X_test_bf_deepseek, y_train, y_test, results)

display_styled_results(results, file_name="AutoFeat-BigFeat-Logistic")

,Method,Accuracy,Precision,Recall,F1-Score
0,LOGISTIC (AutoFeat),0.982456,0.986301,0.976744,0.981151
1,LOGISTIC (BigFeat),0.973684,0.974206,0.969702,0.971863
2,LOGISTIC (BigFeat ChatGPT),0.982456,0.986301,0.976744,0.981151
3,LOGISTIC (BigFeat Gemini),0.982456,0.986301,0.976744,0.981151
4,LOGISTIC (BigFeat Claude),0.973684,0.974206,0.969702,0.971863
5,LOGISTIC (BigFeat DeepSeek),0.982456,0.986301,0.976744,0.981151


In [21]:
# KNN Model

results = pd.DataFrame(columns=["Method", "Accuracy", "Precision", "Recall", "F1-Score"])

results = evaluate_model("knn", "AutoFeat", X_train_af, X_test_af, y_train, y_test, results)
results = evaluate_model("knn", "BigFeat", X_train_bf, X_test_bf, y_train, y_test, results)
results = evaluate_model("knn", "BigFeat ChatGPT", X_train_bf_chatgpt, X_test_bf_chatgpt, y_train, y_test, results)
results = evaluate_model("knn", "BigFeat Gemini", X_train_bf_gemini, X_test_bf_gemini, y_train, y_test, results)
results = evaluate_model("knn", "BigFeat Claude", X_train_bf_claude, X_test_bf_claude, y_train, y_test, results)
results = evaluate_model("knn", "BigFeat DeepSeek", X_train_bf_deepseek, X_test_bf_deepseek, y_train, y_test, results)

display_styled_results(results, file_name="AutoFeat-BigFeat-KNN")

,Method,Accuracy,Precision,Recall,F1-Score
0,KNN (AutoFeat),0.938596,0.947368,0.923190,0.932981
1,KNN (BigFeat),0.956140,0.951623,0.955617,0.953534
2,KNN (BigFeat ChatGPT),0.964912,0.962660,0.962660,0.962660
3,KNN (BigFeat Gemini),0.964912,0.962660,0.962660,0.962660
4,KNN (BigFeat Claude),0.964912,0.962660,0.962660,0.962660
5,KNN (BigFeat DeepSeek),0.964912,0.962660,0.962660,0.962660


In [22]:
# RandomForest Model

results = pd.DataFrame(columns=["Method", "Accuracy", "Precision", "Recall", "F1-Score"])

results = evaluate_model("rf", "AutoFeat", X_train_af, X_test_af, y_train, y_test, results)
results = evaluate_model("rf", "BigFeat", X_train_bf, X_test_bf, y_train, y_test, results)
results = evaluate_model("rf", "BigFeat ChatGPT", X_train_bf_chatgpt, X_test_bf_chatgpt, y_train, y_test, results)
results = evaluate_model("rf", "BigFeat Gemini", X_train_bf_gemini, X_test_bf_gemini, y_train, y_test, results)
results = evaluate_model("rf", "BigFeat Claude", X_train_bf_claude, X_test_bf_claude, y_train, y_test, results)
results = evaluate_model("rf", "BigFeat DeepSeek", X_train_bf_deepseek, X_test_bf_deepseek, y_train, y_test, results)

display_styled_results(results, file_name="AutoFeat-BigFeat-RandomForest")

,Method,Accuracy,Precision,Recall,F1-Score
0,RF (AutoFeat),0.956140,0.955357,0.951032,0.953106
1,RF (BigFeat),0.964912,0.967257,0.958074,0.962302
2,RF (BigFeat ChatGPT),0.964912,0.967257,0.958074,0.962302
3,RF (BigFeat Gemini),0.964912,0.962660,0.962660,0.962660
4,RF (BigFeat Claude),0.973684,0.974206,0.969702,0.971863
5,RF (BigFeat DeepSeek),0.973684,0.974206,0.969702,0.971863


In [23]:
# SVM Model

results = pd.DataFrame(columns=["Method", "Accuracy", "Precision", "Recall", "F1-Score"])

results = evaluate_model("svc", "AutoFeat", X_train_af, X_test_af, y_train, y_test, results)
results = evaluate_model("svc", "BigFeat", X_train_bf, X_test_bf, y_train, y_test, results)
results = evaluate_model("svc", "BigFeat ChatGPT", X_train_bf_chatgpt, X_test_bf_chatgpt, y_train, y_test, results)
results = evaluate_model("svc", "BigFeat Gemini", X_train_bf_gemini, X_test_bf_gemini, y_train, y_test, results)
results = evaluate_model("svc", "BigFeat Claude", X_train_bf_claude, X_test_bf_claude, y_train, y_test, results)
results = evaluate_model("svc", "BigFeat DeepSeek", X_train_bf_deepseek, X_test_bf_deepseek, y_train, y_test, results)

display_styled_results(results, file_name="AutoFeat-BigFeat-SVM")

,Method,Accuracy,Precision,Recall,F1-Score
0,SVC (AutoFeat),0.921053,0.943750,0.895349,0.911757
1,SVC (BigFeat),0.982456,0.986301,0.976744,0.981151
2,SVC (BigFeat ChatGPT),0.973684,0.974206,0.969702,0.971863
3,SVC (BigFeat Gemini),0.973684,0.974206,0.969702,0.971863
4,SVC (BigFeat Claude),0.982456,0.986301,0.976744,0.981151
5,SVC (BigFeat DeepSeek),0.973684,0.974206,0.969702,0.971863


In [24]:
# DecisionTree Model

results = pd.DataFrame(columns=["Method", "Accuracy", "Precision", "Recall", "F1-Score"])

results = evaluate_model("decision_tree", "AutoFeat", X_train_af, X_test_af, y_train, y_test, results)
results = evaluate_model("decision_tree", "BigFeat", X_train_bf, X_test_bf, y_train, y_test, results)
results = evaluate_model("decision_tree", "BigFeat ChatGPT", X_train_bf_chatgpt, X_test_bf_chatgpt, y_train, y_test, results)
results = evaluate_model("decision_tree", "BigFeat Gemini", X_train_bf_gemini, X_test_bf_gemini, y_train, y_test, results)
results = evaluate_model("decision_tree", "BigFeat Claude", X_train_bf_claude, X_test_bf_claude, y_train, y_test, results)
results = evaluate_model("decision_tree", "BigFeat DeepSeek", X_train_bf_deepseek, X_test_bf_deepseek, y_train, y_test, results)

display_styled_results(results, file_name="AutoFeat-BigFeat-DecisionTree")

,Method,Accuracy,Precision,Recall,F1-Score
0,DECISION_TREE (AutoFeat),0.938596,0.936508,0.932362,0.934348
1,DECISION_TREE (BigFeat),0.929825,0.921245,0.934491,0.926570
2,DECISION_TREE (BigFeat ChatGPT),0.921053,0.912724,0.922863,0.917064
3,DECISION_TREE (BigFeat Gemini),0.947368,0.941063,0.948575,0.944481
4,DECISION_TREE (BigFeat Claude),0.964912,0.962660,0.962660,0.962660
5,DECISION_TREE (BigFeat DeepSeek),0.947368,0.941063,0.948575,0.944481


In [30]:
# Save Generated Features

# Save ChatGpt
feature_lines = describe_generated_features(bf_chatgpt)
new_only, original_only = split_original_vs_new(bf_chatgpt, X_train_bf_chatgpt)
save_generated_features(feature_lines, new_only.shape[1], original_only.shape, X_train_bf_chatgpt.shape, "generated_features_chatgpt.txt")

# Save Gemini
feature_lines = describe_generated_features(bf_gemini)
new_only, original_only = split_original_vs_new(bf_gemini, X_train_bf_gemini)
save_generated_features(feature_lines, new_only.shape[1], original_only.shape, X_train_bf_gemini.shape, "generated_features_gemini.txt")

# Save Claude
feature_lines = describe_generated_features(bf_claude)
new_only, original_only = split_original_vs_new(bf_claude, X_train_bf_claude)
save_generated_features(feature_lines, new_only.shape[1], original_only.shape, X_train_bf_claude.shape, "generated_features_claude.txt")

# Save DeepSeek
feature_lines = describe_generated_features(bf_deepseek)
new_only, original_only = split_original_vs_new(bf_deepseek, X_train_bf_deepseek)
save_generated_features(feature_lines, new_only.shape[1], original_only.shape, X_train_bf_deepseek.shape, "generated_features_deepseek.txt")